# Claude Agent SDK functionality primer — before Day 1 PM

This notebook introduces the Claude Agent SDK APIs required by the main Day 1 demonstrations:

- `ClaudeAgentOptions`, `query()`, and `ClaudeSDKClient`
- built-in tools and custom tools through an in-process MCP server
- JSON-Schema output and native message blocks
- `session_id`, `resume`, project settings, and context continuity
- `allowed_tools`, permission modes, and `can_use_tool`
- model selection, turn limits, budget limits, and result metering

Set `ANTHROPIC_API_KEY` in `Week2/.env` before running the provider-backed cells. Domain tools use the small in-code claim and shipment records below.

<img src="claude_agent_sdk_fundamentals.png" width="900" alt="Claude Agent SDK agentic AI fundamentals: entry points, options, agent loop, tools, permissions, sessions, and metering">

In [ ]:
print()

In [6]:
import asyncio
import json
import os
import sys
from collections.abc import Callable, Coroutine
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Any, Literal, TypeVar

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    ClaudeSDKClient,
    ResultMessage,
    TextBlock,
    ThinkingBlock,
    ToolAnnotations,
    ToolResultBlock,
    ToolUseBlock,
    create_sdk_mcp_server,
    query,
    tool,
)
from claude_agent_sdk.types import PermissionResultAllow, PermissionResultDeny, ToolPermissionContext


def discover_week2_root() -> Path:
    """Find Week2 by walking upward from the kernel working directory."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "W2D1" / "Agentic-libraries").is_dir() and (
            candidate / "requirements.txt"
        ).is_file():
            return candidate
    raise RuntimeError("Start the notebook from this repository or one of its subdirectories.")


WEEK2_ROOT = discover_week2_root()
load_dotenv(WEEK2_ROOT / ".env")

T = TypeVar("T")


def run_sdk(factory: Callable[[], Coroutine[Any, Any, T]]) -> T:
    """Run Claude Agent SDK work on a loop that can spawn Claude Code.

    On Windows, Jupyter often uses SelectorEventLoop, which cannot create
    subprocesses. The SDK needs ProactorEventLoop for the Claude Code CLI.
    """

    def _runner() -> T:
        loop = (
            asyncio.ProactorEventLoop()
            if sys.platform == "win32"
            else asyncio.new_event_loop()
        )
        asyncio.set_event_loop(loop)
        try:
            return loop.run_until_complete(factory())
        finally:
            loop.run_until_complete(loop.shutdown_asyncgens())
            loop.close()

    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(_runner).result()


CLAIMS = {
    "CLM-101": {"policy_id": "POL-101", "severity": "low", "status": "open"},
}
SHIPMENTS = {
    "BOX-7": {"status": "held", "reason": "address check"},
}


class TriageDecision(BaseModel):
    """Small structured response used by this primer."""

    case_id: str
    urgency: Literal["low", "high"]
    route: str
    summary: str
    confidence: float = Field(ge=0, le=1)


if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("Set ANTHROPIC_API_KEY in Week2/.env before running this primer.")

print("Primer data ready")
print("Week2 root:", WEEK2_ROOT)
print("Model:", os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6"))

Primer data ready
Week2 root: D:\Ambilio\EXL-CampusHire\Week2
Model: claude-sonnet-4-6


## 1. The Claude Agent SDK

The SDK exposes the same agent loop, tools, and context management used by Claude Code as a Python or TypeScript library.

- `query()` is the smallest entry point for one bounded task.
- `ClaudeSDKClient` supports controlled multi-turn work and interruption.
- `ClaudeAgentOptions` configures the runtime around the model.
- `ResultMessage` reports the final result, session, usage, cost estimate, and stop information.

Run the next two cells to see both entry points.

On Windows Jupyter, call live SDK work through `run_sdk(...)` from the setup cell so Claude Code can start under a Proactor event loop.

In [14]:
simple_options = ClaudeAgentOptions(
    model=os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6"),
    tools=[],
    max_turns=2,
    max_budget_usd=0.10,
)


async def demo_query_raw() -> None:
    """Call query() with a simple prompt and print every returned message."""
    async for message in query(
        prompt="Reply in one short sentence: what is a claim ID?",
        options=simple_options,
    ):
        print(type(message).__name__)
        print(message)
        print("-" * 40)


run_sdk(demo_query_raw)

SystemMessage
SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': 'd:\\Ambilio\\EXL-CampusHire\\Week2\\W2D1\\Agentic-libraries', 'session_id': '67cb5651-2f6c-43b6-bfa2-5db7f37607b6', 'tools': [], 'mcp_servers': [], 'model': 'claude-sonnet-4-6', 'permissionMode': 'default', 'slash_commands': ['deep-research', 'design-sync', 'dataviz', 'update-config', 'verify', 'debug', 'code-review', 'simplify', 'batch', 'fewer-permission-prompts', 'doctor', 'loop', 'claude-api', 'run', 'run-skill-generator', 'agents', 'clear', 'color', 'compact', 'config', 'context', 'effort', 'fast', 'heapdump', 'init', 'mcp', 'model', '__remote-workflow', 'reload-skills', 'rename', 'review', 'security-review', 'usage', 'insights', 'recap', 'goal', 'design', 'design-consent', 'design-revoke', 'team-onboarding'], 'apiKeySource': 'ANTHROPIC_API_KEY', 'claude_code_version': '2.1.215', 'output_style': 'default', 'agents': ['claude', 'Explore', 'general-purpose', 'Plan', 'statusline-setup'], 's

In [7]:
base_options = ClaudeAgentOptions(
    model=os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6"),
    tools=[],
    max_turns=4,
    max_budget_usd=0.10,
)


async def demo_query() -> None:
    """query() — one bounded task; streams messages until ResultMessage."""
    async for message in query(
        prompt=(
            "In two short sentences, explain why an operations agent must look up "
            "supplied claim records instead of inventing claim facts."
        ),
        options=base_options,
    ):
        if isinstance(message, ResultMessage):
            print("query() result:")
            print(message.result)
            print(
                f"session={message.session_id} turns={message.num_turns} "
                f"cost_usd={message.total_cost_usd}"
            )


run_sdk(demo_query)

query() result:
Inventing claim facts introduces fabricated data into decisions that affect real financial payouts, coverage determinations, and legal obligations, potentially causing significant harm to claimants and the organization. An operations agent must retrieve the actual supplied records to ensure every action is grounded in verified, auditable truth that can be traced back to authoritative sources.
session=b20fc69d-2981-403c-8c8b-d2d83da098af turns=1 cost_usd=0.002756


In [8]:
async def demo_client() -> None:
    """ClaudeSDKClient — same session across related follow-up turns."""
    async with ClaudeSDKClient(options=base_options) as client:
        await client.query(
            "Remember this supplied claim fact only: CLM-101 is open with severity low."
        )
        async for message in client.receive_response():
            if isinstance(message, ResultMessage):
                print("Turn 1:", message.result)
                print("session:", message.session_id)

        await client.query("What claim ID and status did I just give you?")
        async for message in client.receive_response():
            if isinstance(message, ResultMessage):
                print("Turn 2:", message.result)
                print(
                    f"turns={message.num_turns} cost_usd={message.total_cost_usd}"
                )


run_sdk(demo_client)

Turn 1: I've noted the following claim fact:

- **Claim ID:** CLM-101
- **Status:** Open
- **Severity:** Low

I'll retain this information for reference during our conversation. How can I assist you further?
session: 8d566466-82f9-4bda-9b53-f596d06c138c
Turn 2: Based on the information you provided, the claim details are:

- **Claim ID:** CLM-101
- **Status:** Open
turns=1 cost_usd=0.0037689999999999998


## 2. Configure built-in and custom tools

Built-in tools are selected by name in `ClaudeAgentOptions.tools`, such as `Read`, `Bash`, and `WebFetch`.

Custom domain tools use `@tool` and `create_sdk_mcp_server()`. With server name `ops`, the runtime-visible names use the form `mcp__ops__<tool>`.

The custom tools below read only the supplied in-code records. The write-shaped tool is registered but is not executed in this primer.

In [9]:
built_in_options = ClaudeAgentOptions(
    model=os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6"),
    tools=["Read", "Bash", "WebFetch"],
    max_turns=4,
)


def mcp_result(payload: object, is_error: bool = False) -> dict[str, Any]:
    """Wrap a Python value as an in-process MCP text result."""
    return {
        "content": [{"type": "text", "text": json.dumps(payload)}],
        "is_error": is_error,
    }


@tool(
    "claim_lookup",
    "Return one supplied claim by claim ID.",
    {"claim_id": str},
    annotations=ToolAnnotations(readOnlyHint=True),
)
async def claim_lookup(args: dict[str, Any]) -> dict[str, Any]:
    """Read one supplied claim record."""
    record = CLAIMS.get(args["claim_id"])
    return mcp_result(record or {"found": False, "claim_id": args["claim_id"]})


@tool(
    "shipment_lookup",
    "Return one supplied shipment by container ID.",
    {"container_id": str},
    annotations=ToolAnnotations(readOnlyHint=True),
)
async def shipment_lookup(args: dict[str, Any]) -> dict[str, Any]:
    """Read one supplied shipment record."""
    record = SHIPMENTS.get(args["container_id"])
    return mcp_result(record or {"found": False, "container_id": args["container_id"]})


RELEASE_LEDGER: dict[str, dict[str, Any]] = {}


@tool(
    "release_hold",
    "Release a shipment hold after approval; requires an idempotency key.",
    {"container_id": str, "approved": bool, "idempotency_key": str},
    annotations=ToolAnnotations(readOnlyHint=False, destructiveHint=True),
)
async def release_hold(args: dict[str, Any]) -> dict[str, Any]:
    """Return an idempotent receipt for an approved hold release."""
    key = args["idempotency_key"].strip()
    if not args["approved"] or not key:
        return mcp_result({"executed": False, "reason": "approval and idempotency key required"}, True)
    if args["container_id"] not in SHIPMENTS:
        return mcp_result({"executed": False, "reason": "shipment not found"}, True)
    if key in RELEASE_LEDGER:
        return mcp_result({**RELEASE_LEDGER[key], "duplicate": True})
    receipt = {"executed": True, "container_id": args["container_id"], "idempotency_key": key}
    RELEASE_LEDGER[key] = receipt
    return mcp_result(receipt)


ops_server = create_sdk_mcp_server(
    name="ops",
    version="1.0.0",
    tools=[claim_lookup, shipment_lookup, release_hold],
)
print("Built-in tools:", built_in_options.tools)
print("Custom server registered as: ops")

Built-in tools: ['Read', 'Bash', 'WebFetch']
Custom server registered as: ops


## 3. Configure structured output and inspect messages

`output_format` accepts a JSON Schema. The final `ResultMessage.structured_output` can then be validated with the same Pydantic model.

An `AssistantMessage` may contain `TextBlock`, `ToolUseBlock`, `ToolResultBlock`, or `ThinkingBlock` values. The helper below inspects the SDK's message objects without making a request.

In [10]:
structured_options = ClaudeAgentOptions(
    model=os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6"),
    tools=[],
    mcp_servers={"ops": ops_server},
    output_format={"type": "json_schema", "schema": TriageDecision.model_json_schema()},
    max_turns=4,
)


def inspect_sdk_message(message: object) -> dict[str, Any]:
    """Return the SDK message type and its contained block types."""
    if isinstance(message, AssistantMessage):
        return {
            "message_type": type(message).__name__,
            "block_types": [type(block).__name__ for block in message.content],
        }
    if isinstance(message, ResultMessage):
        return {
            "message_type": type(message).__name__,
            "session_id": message.session_id,
            "structured_output": message.structured_output,
        }
    return {"message_type": type(message).__name__}


print("Output schema:", structured_options.output_format)
print("Recognized blocks:", [
    TextBlock.__name__,
    ToolUseBlock.__name__,
    ToolResultBlock.__name__,
    ThinkingBlock.__name__,
])

Output schema: {'type': 'json_schema', 'schema': {'description': 'Small structured response used by this primer.', 'properties': {'case_id': {'title': 'Case Id', 'type': 'string'}, 'urgency': {'enum': ['low', 'high'], 'title': 'Urgency', 'type': 'string'}, 'route': {'title': 'Route', 'type': 'string'}, 'summary': {'title': 'Summary', 'type': 'string'}, 'confidence': {'maximum': 1, 'minimum': 0, 'title': 'Confidence', 'type': 'number'}}, 'required': ['case_id', 'urgency', 'route', 'summary', 'confidence'], 'title': 'TriageDecision', 'type': 'object'}}
Recognized blocks: ['TextBlock', 'ToolUseBlock', 'ToolResultBlock', 'ThinkingBlock']


## 4. Configure project context and session resume

`ClaudeSDKClient` keeps context while the client remains connected. `ResultMessage.session_id` identifies the session; pass that value to `ClaudeAgentOptions.resume` to continue it later.

`cwd` and `setting_sources=["project"]` enable project instructions such as `CLAUDE.md` or supported `AGENTS.md` context. The runtime manages context compaction during longer sessions.

In [11]:
PROJECT_ROOT = Path("../W2D1PM")


def session_options(resume_id: str | None = None) -> ClaudeAgentOptions:
    """Create SDK options for a new or resumed project-aware session."""
    return ClaudeAgentOptions(
        model=os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6"),
        tools=[],
        mcp_servers={"ops": ops_server},
        cwd=PROJECT_ROOT,
        setting_sources=["project"],
        resume=resume_id,
        max_turns=4,
    )


new_session_options = session_options()
resumed_session_options = session_options("example-session-id")

print("Project root:", new_session_options.cwd)
print("Setting sources:", new_session_options.setting_sources)
print("Resume ID:", resumed_session_options.resume)

Project root: ..\W2D1PM
Setting sources: ['project']
Resume ID: example-session-id


## 5. Configure permissions and model selection

- `allowed_tools` auto-approves matching tools.
- `permission_mode` selects the SDK's broader permission behavior.
- `can_use_tool` supplies an application callback for calls that require a decision.
- `model` selects Sonnet 4.6 or Opus 4.8 independently of permissions.

The callback below denies the write because learner approval and a matching idempotency key have not been supplied.

In [12]:
READ_TOOLS = [
    "mcp__ops__claim_lookup",
    "mcp__ops__shipment_lookup",
]
WRITE_TOOL = "mcp__ops__release_hold"
LEARNER_APPROVED = False
APPROVED_KEY = ""


async def permission_gate(
    tool_name: str,
    input_data: dict[str, Any],
    context: ToolPermissionContext,
) -> PermissionResultAllow | PermissionResultDeny:
    """Allow the configured write only after explicit learner approval."""
    approved_write = (
        tool_name == WRITE_TOOL
        and LEARNER_APPROVED
        and bool(APPROVED_KEY)
        and input_data.get("idempotency_key") == APPROVED_KEY
    )
    if approved_write:
        return PermissionResultAllow(updated_input=input_data)
    return PermissionResultDeny(
        message="Write requires learner approval and a matching idempotency key."
    )


sonnet_options = ClaudeAgentOptions(
    model="claude-sonnet-4-6",
    tools=[],
    mcp_servers={"ops": ops_server},
    allowed_tools=READ_TOOLS,
    can_use_tool=permission_gate,
    permission_mode="default",
    max_turns=4,
)
opus_options = ClaudeAgentOptions(
    model="claude-opus-4-8",
    tools=[],
    max_turns=4,
)

print("Auto-approved reads:", sonnet_options.allowed_tools)
print("Permission mode:", sonnet_options.permission_mode)
print("Configured models:", sonnet_options.model, opus_options.model)

Auto-approved reads: ['mcp__ops__claim_lookup', 'mcp__ops__shipment_lookup']
Permission mode: default
Configured models: claude-sonnet-4-6 claude-opus-4-8


## 6. Configure limits and read metering

`ClaudeAgentOptions.max_turns` limits agent turns. `max_budget_usd` sets the SDK's client-estimated budget cap.

The final `ResultMessage` exposes `num_turns`, `usage`, `model_usage`, `total_cost_usd`, `stop_reason`, and `subtype`. The helper below extracts those SDK fields from a real result when the main demonstrations run.

In [13]:
budgeted_options = ClaudeAgentOptions(
    model="claude-sonnet-4-6",
    tools=[],
    max_turns=4,
    max_budget_usd=0.10,
)


def result_metering(result: ResultMessage) -> dict[str, Any]:
    """Extract usage and stop information from an SDK ResultMessage."""
    return {
        "session_id": result.session_id,
        "num_turns": result.num_turns,
        "usage": result.usage,
        "model_usage": result.model_usage,
        "total_cost_usd": result.total_cost_usd,
        "stop_reason": result.stop_reason,
        "subtype": result.subtype,
    }


print("Turn limit:", budgeted_options.max_turns)
print("Budget limit:", budgeted_options.max_budget_usd)
print("result_metering() is ready for a ResultMessage from the main demonstrations.")

Turn limit: 4
Budget limit: 0.1
result_metering() is ready for a ResultMessage from the main demonstrations.


## Continue to the main PM demonstrations

This primer used the Claude Agent SDK surfaces required for Day 1: options, one-shot and client entry points, built-in tools, custom MCP tools, structured output, message inspection, session resume, project settings, permissions, model selection, limits, and metering fields.

**Next:** open `../W2D1PM/_DEMONSTRATIONS.ipynb` to execute these APIs with the configured Claude runtime.